In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import copy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
csv_path = "/content/drive/My Drive/pcos_rotterdam_balanceado.csv"
label_column = "PCOS_Diagnosis"

num_clients = 10
communication_rounds = 20
batch_size = 128
learning_rate = 0.0001
test_size = 0.2

In [ ]:
# Load CSV
data = pd.read_csv(csv_path)
y = data[label_column].values
X = data.drop(columns=[label_column]).values

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

# Check class balance after SMOTE
print("Class distribution after SMOTE:\n", pd.Series(y_smote).value_counts())

Class distribution after SMOTE:
 0    2400
1    2400
Name: count, dtype: int64


In [ ]:
# Split global train-test
X_train_global, X_test, y_train_global, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=42
)

# Stratified split into clients
skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=42)
client_datasets = []
for _, idx in skf.split(X_train_global, y_train_global):
    client_datasets.append((X_train_global[idx], y_train_global[idx]))

In [ ]:
from tensorflow.keras.layers import Input
# MLP Model
def create_mlp(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)), # Explicit Input layer to avoid warning
        Dense(128, activation="relu"),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=BinaryCrossentropy(),
        metrics=["accuracy"]
    )
    return model

In [ ]:
# FedAvg aggregation
def fedavg(global_weights, client_weights):
    new_weights = []
    for weights_list_tuple in zip(*client_weights):
        new_weights.append(
            np.mean(np.array(weights_list_tuple, dtype=object), axis=0)
        )
    return new_weights

In [ ]:
# Initialize global model
global_model = create_mlp(X.shape[1])

history = []

# 🔁 Communication rounds
for rnd in range(communication_rounds):
    print(f"\n🔁 Round {rnd+1}/{communication_rounds}")

    client_weights = []

    # Train each client locally
    for cid, (Xc, yc) in enumerate(client_datasets):
        client_model = create_mlp(X.shape[1])
        client_model.set_weights(global_model.get_weights())

        client_model.fit(
            Xc, yc,
            epochs=3,        # small — prevents divergence
            batch_size=batch_size,
            verbose=0
        )
        client_weights.append(client_model.get_weights())

    # FedAvg aggregation
    new_global_weights = fedavg(global_model.get_weights(), client_weights)
    global_model.set_weights(new_global_weights)

    # Evaluate
    loss, acc = global_model.evaluate(X_test, y_test, verbose=0)
    history.append(acc)

    print(f"📌 Global loss: {loss:.4f} - Global accuracy: {acc:.4f}")

# Save the final model after all rounds
global_model.save('federated_mlp_model.keras')


🔁 Round 1/20
📌 Global loss: 0.6403 - Global accuracy: 0.8350

🔁 Round 2/20
📌 Global loss: 0.6203 - Global accuracy: 0.8950

🔁 Round 3/20
📌 Global loss: 0.6007 - Global accuracy: 0.9350

🔁 Round 4/20
📌 Global loss: 0.5816 - Global accuracy: 0.9717

🔁 Round 5/20
📌 Global loss: 0.5628 - Global accuracy: 0.9817

🔁 Round 6/20
📌 Global loss: 0.5444 - Global accuracy: 0.9817

🔁 Round 7/20
📌 Global loss: 0.5263 - Global accuracy: 0.9817

🔁 Round 8/20
📌 Global loss: 0.5086 - Global accuracy: 0.9867

🔁 Round 9/20
📌 Global loss: 0.4911 - Global accuracy: 0.9900

🔁 Round 10/20
📌 Global loss: 0.4739 - Global accuracy: 0.9900

🔁 Round 11/20
📌 Global loss: 0.4571 - Global accuracy: 0.9917

🔁 Round 12/20
📌 Global loss: 0.4406 - Global accuracy: 0.9917

🔁 Round 13/20
📌 Global loss: 0.4245 - Global accuracy: 0.9917

🔁 Round 14/20
📌 Global loss: 0.4088 - Global accuracy: 0.9917

🔁 Round 15/20
📌 Global loss: 0.3934 - Global accuracy: 0.9933

🔁 Round 16/20
📌 Global loss: 0.3783 - Global accuracy: 0.9933



In [ ]:
OUTPUT_WEIGHTS_FILE = "/content/drive/MyDrive/PCOS MINOR/MLP FedAvg (CSV).keras"
global_model.save(OUTPUT_WEIGHTS_FILE)
print(f"\n✅ PSO-FedAvg complete! Model saved to {OUTPUT_WEIGHTS_FILE}")


✅ PSO-FedAvg complete! Model saved to /content/drive/MyDrive/PCOS MINOR/MLP FedAvg (CSV).keras
